# Using `pyologger` data processing pipeline with `DiveDB`
Uses classes `Metadata` and `DataReader` to facilitate data intake, processing, and alignment. 

## Read deployment metadata

In [ ]:
import re
# Import pyologger utilities
from pyologger.utils.folder_manager import *
from pyologger.plot_data.plotter import *
from pyologger.utils.param_manager import ParamManager
from pyologger.load_data.datareader import DataReader
from pyologger.load_data.metadata import Metadata

# Load important file paths and configurations
config, data_dir, color_mapping_path, montage_path = load_configuration()

In [ ]:
data_dir

## Clearing out previous outputs

This script will allow you to delete all previous processed output data at the dataset level. Only use if you are confident you can re-generate the output data with the raw data on hand.

While in `pyologger` root directory:

Preview what will be deleted / moved to trash:
```python3 scripts/delete_outputs.py --dataset <DATASET-ID> --dry-run```
```python3 scripts/delete_outputs.py --dataset pale-adult-lion_vid-accel_africa_TW --dry-run```

Move to trash (quite slow but intentional :)
```python3 scripts/delete_outputs.py --dataset pale-adult-lion_vid-accel_africa_TW --trash```

Empty trash:
```python3 scripts/delete_outputs.py --empty-trash```

Quickly delete permanently:
```python3 scripts/delete_outputs.py --dataset pale-adult-lion_vid-accel_africa_TW --delete```

Remove folder(s) as well:
```python3 scripts/delete_outputs.py --dataset pale-adult-lion_vid-accel_africa_TW --remove-folder```

SANDBOX: 
```{bash}
python3 scripts/delete_outputs.py --dataset mile-adult-sese_vdr_argentina_RD-KM --delete
```

List of Dataset IDs:
- mile-adult-sese_vdr_argentina_RD-KM
- mian-adult-nese_tdr-sr_TA-AT-YN-DC
- mian-juv-nese_sleep_lml-ano_JKB
- pale-adult-lion_vid-accel_africa_TW

python3 workflows/06_export_data.py \
  --dataset oror-adult-orca_hr-sr-vid_sw_JKB-PP \
  --deployment 2024-12-19_oror-001 \
  --csvs-only


### Fetch metadata

Load in metadata stored in Notion databases. Alternatively, load in your own metadata in separate dataframes for deployments, loggers, recordings, animals, and datasets. See examples here in the `metadata_snapshot.pkl` file.

In [ ]:
import os
import json
import pickle
import tempfile
from datetime import datetime, timedelta

# ---- user-provided context assumed ----
# data_dir: base data directory
# config: dict with paths (e.g., config["paths"]["local_repo_path"])
# Metadata: the class you just updated

overwrite = False  # Force refresh from Notion if True

# Paths
metadata_dir = os.path.join(data_dir, "00_Metadata")
os.makedirs(metadata_dir, exist_ok=True)
metadata_pickle_path = os.path.join(metadata_dir, "metadata_snapshot.pkl")

relations_map_dir = config["paths"].get("local_repo_path", metadata_dir)
os.makedirs(relations_map_dir, exist_ok=True)
relations_map_path = os.path.join(relations_map_dir, "relations_map.json")


def atomic_write_bytes(path: str, data: bytes):
    """Write bytes atomically to avoid partial/corrupt files."""
    dirpath = os.path.dirname(path)
    os.makedirs(dirpath, exist_ok=True)
    with tempfile.NamedTemporaryFile(dir=dirpath, delete=False) as tmp:
        tmp.write(data)
        tmp.flush()
        os.fsync(tmp.fileno())
        tmp_path = tmp.name
    os.replace(tmp_path, path)


def atomic_write_text(path: str, text: str):
    atomic_write_bytes(path, text.encode("utf-8"))


def load_all_tables(md_obj):
    """
    Convenience: pull all the DB/data source tables from a Metadata instance.
    Returns a dict so you can unpack if you want.
    """
    return {
        "deployment_db": md_obj.get_metadata("deployment_DB"),
        "logger_db": md_obj.get_metadata("logger_DB"),
        "recording_db": md_obj.get_metadata("recording_DB"),
        "animal_db": md_obj.get_metadata("animal_DB"),
        "dataset_db": md_obj.get_metadata("dataset_DB"),
        "procedure_db": md_obj.get_metadata("procedure_DB"),
        "observation_db": md_obj.get_metadata("observation_DB"),
        "collaborator_db": md_obj.get_metadata("collaborator_DB"),
        "location_db": md_obj.get_metadata("location_DB"),
        "montage_db": md_obj.get_metadata("montage_DB"),
        "signal_db": md_obj.get_metadata("signal_DB"),
        "attachment_db": md_obj.get_metadata("attachment_DB"),
        "originalchannel_db": md_obj.get_metadata("originalchannel_DB"),
        "standardizedchannel_db": md_obj.get_metadata("standardizedchannel_DB")
    }


def pickle_needs_refresh(path: str, max_age_days: int = 14) -> bool:
    if not os.path.exists(path):
        return True
    try:
        mtime = datetime.fromtimestamp(os.path.getmtime(path))
        return (datetime.now() - mtime) > timedelta(days=max_age_days)
    except Exception:
        # If anything is weird with the file, refresh.
        return True


# Decide whether to pull fresh data from Notion
needs_refresh = overwrite or pickle_needs_refresh(metadata_pickle_path, max_age_days=14)

if needs_refresh:
    # 1) Build a fresh Metadata instance (hits Notion and populates self.metadata etc.)
    metadata = Metadata()

    # 2) Recompute relations map (uses databases.retrieve schemas and relation fields)
    metadata.map_database_relations()
    relations_map = metadata.relations_map

    # 3) Save relations map atomically (so it’s never half-written)
    try:
        atomic_write_text(relations_map_path, json.dumps(relations_map, indent=4))
        print(f"Relations map saved at: {relations_map_path}")
    except Exception as e:
        print(f"[WARN] Failed to write relations_map.json: {e}")

    # 4) Strip live client / transient runtime caches before pickling
    metadata.notion = None
    if hasattr(metadata, "data_source_cache"):
        metadata.data_source_cache = {}

    # Optional: embed a tiny snapshot header for sanity
    snapshot_meta = {
        "notion_version": getattr(metadata, "notion_version", None),
        "created_at": datetime.now().isoformat(),
        "class": "Metadata",
    }
    payload = {"snapshot_meta": snapshot_meta, "metadata_obj": metadata}

    # 5) Snapshot full metadata object to disk atomically
    try:
        atomic_write_bytes(metadata_pickle_path, pickle.dumps(payload, protocol=pickle.HIGHEST_PROTOCOL))
        print(f"[REFRESH] Metadata snapshot saved at: {metadata_pickle_path}")
    except Exception as e:
        print(f"[ERROR] Failed to write metadata snapshot: {e}")
        raise
else:
    # Load cached snapshot instead of hitting Notion
    print(f"[CACHE] Using existing metadata snapshot at: {metadata_pickle_path}")
    try:
        with open(metadata_pickle_path, "rb") as file:
            payload = pickle.load(file)
        # Backward-compat: support old format (raw Metadata pickled directly)
        if isinstance(payload, dict) and "metadata_obj" in payload:
            metadata = payload["metadata_obj"]
            snapshot_meta = payload.get("snapshot_meta", {})
        else:
            metadata = payload
            snapshot_meta = {}
        # Note: metadata.notion is None here (by design). We're only reading dfs, so it's fine.
    except Exception as e:
        print(f"[WARN] Cache unreadable ({e}). Falling back to fresh pull.")
        metadata = Metadata()
        metadata.map_database_relations()
        relations_map = metadata.relations_map
        try:
            atomic_write_text(relations_map_path, json.dumps(relations_map, indent=4))
        except Exception as ee:
            print(f"[WARN] Failed to write relations_map.json on fallback: {ee}")
        metadata.notion = None
        if hasattr(metadata, "data_source_cache"):
            metadata.data_source_cache = {}
        payload = {
            "snapshot_meta": {
                "notion_version": getattr(metadata, "notion_version", None),
                "created_at": datetime.now().isoformat(),
                "class": "Metadata",
            },
            "metadata_obj": metadata,
        }
        atomic_write_bytes(metadata_pickle_path, pickle.dumps(payload, protocol=pickle.HIGHEST_PROTOCOL))
        print(f"[REFRESH] Metadata snapshot saved at: {metadata_pickle_path}")


# Expose each table for downstream code in this session
tables = load_all_tables(metadata)

deployment_db = tables["deployment_db"]
logger_db = tables["logger_db"]
recording_db = tables["recording_db"]
animal_db = tables["animal_db"]
dataset_db = tables["dataset_db"]
procedure_db = tables["procedure_db"]
observation_db = tables["observation_db"]
collaborator_db = tables["collaborator_db"]
location_db = tables["location_db"]
montage_db = tables["montage_db"]
signal_db = tables["signal_db"]
attachment_db = tables["attachment_db"]
originalchannel_db = tables["originalchannel_db"]
standardizedchannel_db = tables["standardizedchannel_db"]

# Optional: quick sanity print of row counts
try:
    counts = {k: (v.shape[0] if v is not None else 0) for k, v in tables.items()}
    print("[Metadata tables] row counts:", json.dumps(counts, indent=2))
except Exception:
    pass


In [ ]:
print(procedure_db)

In [ ]:
import json
import os

# Path to write JSON export
json_export_path = os.path.join(metadata_dir, "metadata_snapshot.json")

def df_to_records_safe(df):
    """Convert DataFrame to list of records, making all values JSON serializable."""
    return json.loads(df.to_json(orient="records", date_format="iso"))

# Convert all tables to dict of records
json_dict = {}
for name, df in tables.items():
    try:
        json_dict[name] = df_to_records_safe(df)
    except Exception as e:
        print(f"[WARN] Could not convert {name}: {e}")
        json_dict[name] = []

# Save as pretty JSON
with open(json_export_path, "w", encoding="utf-8") as f:
    json.dump(json_dict, f, indent=2)

print(f"✅ Metadata exported to {json_export_path}")


### Select Deployment

In [ ]:
# Select dataset folder
dataset_folder = select_folder(data_dir, "Select a dataset folder:")

In [ ]:
deployment_folder = select_folder(dataset_folder, "Select a deployment folder:")

In [ ]:
# Extract deployment_id and animal_id from the folder name
match = re.match(r"(\d{4}-\d{2}-\d{2}_[a-z]{4}-\d{3})", os.path.basename(deployment_folder), re.IGNORECASE)
if match:
    deployment_id = match.group(1)  # Extract YYYY-MM-DD_animalID
    animal_id = deployment_id.split("_")[1]  # Extract animal ID
    print(f"✅ Extracted deployment ID: {deployment_id}, Animal ID: {animal_id}")
else:
    raise ValueError(f"❌ Unable to extract deployment ID from folder: {deployment_folder}")

In [ ]:
deployment_db

## Read Files in Deployment Folder

Uses [`datareader`](../pyologger/load_data/datareader.py) class and its `read_files()` method to load and standardize data from a deployment folder, map onto standardized channel names, and save as a `data_pkl` object (instance of the datareader class).

In [ ]:
# Print extracted values for debugging
print(f"🐳 Deployment ID: {deployment_id}, Animal ID: {animal_id}")

deployment_info, loggers_used = metadata.extract_essential_metadata(deployment_id)

In [ ]:
import copy
import pytz
overwrite_essential_metadata = False
manual_procedure_overrides_enabled = False

if overwrite_essential_metadata:  # If you store your metadata differently, you can set this manually.
    deployment_id = "2019-11-08_apfo-001"
    animal_id = "apfo-001"
    print(f"🔍 Manually setting essential metadata for Deployment ID: {deployment_id}")

    deployment_info = {
        "Deployment Date": "2019-11-08",
        "Deployment Latitude": -77.858933,
        "Deployment Longitude": 166.5139,
        "Time Zone": "Antarctica/McMurdo",
        "Procedure Info": {}
    }
    print(f"📍 Deployment Metadata: {deployment_info}")

    loggers_used = [
        {"Logger ID": "CC-35", "Manufacturer": "CATS", "Montage ID": "cats-penguin-video-montage_V1"}
    ]
    print(f"📟 Loggers Used: {loggers_used}")

# Manual per-procedure overrides for notebook testing.
# Set manual_procedure_overrides_enabled = True to apply these before read_files().
manual_procedure_overrides = {
    # "2024-10-21_pale-001_attachment": {
    #     "mass": 170,
    #     "mass_kg": 170,
    #     "mass_estimated_kg": 172,
    #     "start_datetime": "2024-10-21T10:00:00",
    #     "end_datetime": "2024-10-21T11:00:00",
    #     "location_name": "Example Site",
    #     "location_latitude": -24.1234,
    #     "location_longitude": 31.5678,
    # }
}

def apply_manual_procedure_overrides(deployment_info, procedure_overrides=None):
    deployment_info = copy.deepcopy(deployment_info)
    procedure_info = deployment_info.setdefault("Procedure Info", {})
    for procedure_id, overrides in (procedure_overrides or {}).items():
        existing = dict(procedure_info.get(procedure_id, {}))
        for key, value in overrides.items():
            if value is not None:
                existing[key] = value
        existing.setdefault("id", procedure_id)
        procedure_info[procedure_id] = existing
    deployment_info["Procedure Info"] = procedure_info
    return deployment_info

def list_available_timezones():
    """Prints all available time zones in pytz."""
    timezones = pytz.all_timezones
    print("\n🌍 Available Time Zones in pytz:\n")
    for tz in timezones:
        print(tz)

# list_available_timezones()
print("Manual procedure override IDs:", list(manual_procedure_overrides.keys()))

In [ ]:
deployment_folder

In [ ]:
# Step 4: Initialize DataReader with dataset folder, deployment ID, and optional data subfolder
data_pkl = DataReader(dataset_folder=dataset_folder, deployment_id=deployment_id, data_subfolder="01_raw-data", montage_path=montage_path)
# Step 5: Initialize config manager
param_manager = ParamManager(deployment_folder=deployment_folder, deployment_id=deployment_id)
param_manager.add_to_config("current_processing_step", "Processing Step 00: Data import pending.")

In [ ]:
loggers_used

In [ ]:
param_manager.export_config()

In [ ]:
overwrite_data = True
pkl_path = os.path.join(deployment_folder, "outputs", "data.pkl")

effective_deployment_info = deployment_info
if manual_procedure_overrides_enabled:
    effective_deployment_info = apply_manual_procedure_overrides(
        deployment_info,
        manual_procedure_overrides,
    )
    print("🧪 Applied manual procedure overrides for:", list(manual_procedure_overrides.keys()))
else:
    print("ℹ️ Using procedure metadata extracted from Notion without notebook overrides.")

if os.path.exists(pkl_path) and not overwrite_data:
    with open(pkl_path, "rb") as f:
        data_pkl = pickle.load(f)
    print(f"📦 Loaded processed DataReader object from: {pkl_path}")
else:
    # Create outputs folder before initializing DataReader
    os.makedirs(os.path.join(deployment_folder, 'outputs'), exist_ok=True)
    
    # Initialize DataReader - it will automatically find deployment folder with suffix
    data_pkl = DataReader(
        dataset_folder=dataset_folder,
        deployment_id=deployment_id,
        data_subfolder="01_raw-data",
        montage_path=montage_path
    )
    
    # Initialize config manager
    param_manager = ParamManager(deployment_folder=deployment_folder, deployment_id=deployment_id)
    param_manager.add_to_config("current_processing_step", "Processing Step 00: Data import pending.")

    # Read and process all files
    data_pkl.read_files(
        deployment_info=effective_deployment_info,
        loggers_used=loggers_used,
        save_parq=False,
        save_netcdf=True
    )


In [ ]:
data_pkl.signal_data['prh']

In [ ]:
data_pkl.signal_data['accelerometer']

## Interactive Dashboard: GPS Track + Accelerometer Timeseries

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from plotly_resampler import FigureResampler
import numpy as np

# Get the data
location_df = data_pkl.signal_data.get('location', pd.DataFrame())
accel_df = data_pkl.signal_data.get('accelerometer', pd.DataFrame())

print(f"Location data: {len(location_df)} points")
print(f"Accelerometer data: {len(accel_df)} points")

if not location_df.empty and not accel_df.empty:
    # Ensure datetime columns are timezone-aware and aligned
    if 'datetime' in location_df.columns:
        location_df['datetime'] = pd.to_datetime(location_df['datetime'])
    if 'datetime' in accel_df.columns:
        accel_df['datetime'] = pd.to_datetime(accel_df['datetime'])
    
    # Create base figure with subplots
    base_fig = make_subplots(
        rows=2, cols=1,
        row_heights=[0.6, 0.4],
        subplot_titles=('GPS Track', 'Accelerometer Timeseries (ax, ay, az)'),
        specs=[[{"type": "scattermapbox"}], [{"type": "scatter"}]],
        vertical_spacing=0.1
    )
    
    # Wrap with FigureResampler for efficient time-series handling
    fig = FigureResampler(base_fig, default_n_shown_samples=2000)
    
    # Add GPS track on map (not resampled - maps handle this differently)
    fig.add_trace(
        go.Scattermapbox(
            lat=location_df['latitude'],
            lon=location_df['longitude'],
            mode='markers+lines',
            marker=dict(size=8, color='blue', opacity=0.7),
            line=dict(width=2, color='lightblue'),
            text=location_df['datetime'].astype(str) if 'datetime' in location_df.columns else None,
            hovertemplate='<b>Lat:</b> %{lat:.6f}<br><b>Lon:</b> %{lon:.6f}<br><b>Time:</b> %{text}<extra></extra>',
            name='GPS Track',
            showlegend=True
        ),
        row=1, col=1
    )
    
    # Add start and end markers
    fig.add_trace(
        go.Scattermapbox(
            lat=[location_df['latitude'].iloc[0]],
            lon=[location_df['longitude'].iloc[0]],
            mode='markers',
            marker=dict(size=15, color='green', symbol='circle'),
            text=['Start'],
            hovertemplate='<b>Start Point</b><br>Lat: %{lat:.6f}<br>Lon: %{lon:.6f}<extra></extra>',
            name='Start',
            showlegend=True
        ),
        row=1, col=1
    )
    
    fig.add_trace(
        go.Scattermapbox(
            lat=[location_df['latitude'].iloc[-1]],
            lon=[location_df['longitude'].iloc[-1]],
            mode='markers',
            marker=dict(size=15, color='red', symbol='circle'),
            text=['End'],
            hovertemplate='<b>End Point</b><br>Lat: %{lat:.6f}<br>Lon: %{lon:.6f}<extra></extra>',
            name='End',
            showlegend=True
        ),
        row=1, col=1
    )
    
    # Add accelerometer traces (ax, ay, az) with resampling for performance
    colors = {'ax': 'red', 'ay': 'green', 'az': 'blue'}
    for axis in ['ax', 'ay', 'az']:
        if axis in accel_df.columns:
            # Use add_trace with hf_x and hf_y for automatic resampling
            fig.add_trace(
                go.Scattergl(  # Use Scattergl for better performance
                    name=axis,
                    line=dict(width=1, color=colors[axis]),
                    opacity=0.7,
                ),
                hf_x=accel_df['datetime'],
                hf_y=accel_df[axis],
                row=2, col=1
            )
    
    # Update map layout
    center_lat = location_df['latitude'].mean()
    center_lon = location_df['longitude'].mean()
    
    fig.update_mapboxes(
        style="open-street-map",
        center=dict(lat=center_lat, lon=center_lon),
        zoom=12
    )
    
    # Update layout
    fig.update_layout(
        height=1000,
        title_text=f"Animal Track & Accelerometer Data - {deployment_id}",
        showlegend=True,
        hovermode='closest',
        legend=dict(
            orientation="v",
            yanchor="top",
            y=0.99,
            xanchor="left",
            x=0.01,
            bgcolor="rgba(255,255,255,0.8)"
        )
    )
    
    # Update x-axis for timeseries
    fig.update_xaxes(title_text="Time", row=2, col=1)
    fig.update_yaxes(title_text="Acceleration (g)", row=2, col=1)
    
    # Show with resampler controls
    fig.show_dash(mode='inline')
    
    # Print summary statistics
    print(f"\n📊 Summary Statistics:")
    print(f"  GPS Track:")
    print(f"    Total points: {len(location_df)}")
    print(f"    Lat range: {location_df['latitude'].min():.6f} to {location_df['latitude'].max():.6f}")
    print(f"    Lon range: {location_df['longitude'].min():.6f} to {location_df['longitude'].max():.6f}")
    if 'datetime' in location_df.columns:
        print(f"    Time range: {location_df['datetime'].min()} to {location_df['datetime'].max()}")
    
    print(f"\n  Accelerometer:")
    print(f"    Total samples: {len(accel_df):,}")
    if 'datetime' in accel_df.columns:
        duration = accel_df['datetime'].max() - accel_df['datetime'].min()
        print(f"    Time range: {accel_df['datetime'].min()} to {accel_df['datetime'].max()}")
        print(f"    Duration: {duration}")
        # Estimate sampling rate
        if len(accel_df) > 1:
            time_diff = (accel_df['datetime'].iloc[-1] - accel_df['datetime'].iloc[0]).total_seconds()
            est_fs = len(accel_df) / time_diff if time_diff > 0 else 0
            print(f"    Estimated sampling rate: {est_fs:.2f} Hz")
    for axis in ['ax', 'ay', 'az']:
        if axis in accel_df.columns:
            print(f"    {axis}: mean={accel_df[axis].mean():.3f}g, std={accel_df[axis].std():.3f}g, range=[{accel_df[axis].min():.3f}, {accel_df[axis].max():.3f}]")
    
    print(f"\n✨ Using plotly-resampler: Displaying ~2000 points per trace (out of {len(accel_df):,} total)")
    print(f"   Zoom in to see more detail - resampler will dynamically load data!")
else:
    print("❌ Missing data: Need both 'location' and 'accelerometer' in signal_data")
    if location_df.empty:
        print("   - No location data found")
    if accel_df.empty:
        print("   - No accelerometer data found")

In [ ]:
data_pkl.signal_info['pressure']['metadata']


In [ ]:
data_pkl.signal_data['accelerometer']

In [ ]:
data_pkl.signal_data['depth']

In [ ]:
data_pkl.signal_info

In [ ]:
data_pkl.logger_info

In [ ]:
from pprint import pprint

def find_unmapped_channels(data_pkl):
    """Return a dict of {signal_name: [channels without parent_signal]}."""
    unmapped = {}

    for signal_name, sig_info in data_pkl.signal_info.items():
        bad_cols = []
        for channel_id, meta in sig_info.get("metadata", {}).items():
            parent = (meta or {}).get("parent_signal")
            if not parent:  # None / "" / missing
                bad_cols.append({"channel": channel_id, "metadata": meta})
        if bad_cols:
            unmapped[signal_name] = bad_cols

    return unmapped

unmapped = find_unmapped_channels(data_pkl)
if unmapped:
    print("⚠️ Channels missing parent_signal:")
    pprint(unmapped)
else:
    print("✅ All mapped channels have a parent_signal.")



In [ ]:
max(data_pkl.signal_data['accelerometer']['datetime'])

In [ ]:
max(data_pkl.signal_data['ecg']['datetime'])

In [ ]:
data_pkl.event_data

In [ ]:
data_pkl.event_data

In [ ]:
data_pkl.signal_info

In [ ]:
data_pkl.signal_data

In [ ]:
data_pkl.animal_info

In [ ]:
data_pkl.signal_info

In [ ]:
data_pkl.logger_info

In [ ]:
data_pkl.animal_info

In [ ]:
data_pkl.event_data

In [ ]:
data_pkl.signal_info

In [ ]:
import pandas as pd
from datetime import timedelta

# Get timezone
timezone = data_pkl.deployment_info.get("Time Zone", "UTC")
replace = True

# Load time settings
time_settings = param_manager.get_from_config(
    ["overlap_start_time", "overlap_end_time", "zoom_window_start_time", "zoom_window_end_time"],
    section="settings"
)

if time_settings and replace == False:
    print("Time settings present.")
# If any required time settings are missing, compute and update them
if not any(v is None for v in time_settings.values()) and replace == False:
    print("Time settings not empty.")
else:
    print("Adding timestamps to config.")
    zoom_time_window = 5  # minutes

    # Extract start and end times for all signals (normalize tz-naive/tz-aware first)
    start_times = []
    end_times = []

    for signal_name, df in data_pkl.signal_data.items():
        if 'datetime' not in df.columns or df.empty:
            continue

        dt = pd.to_datetime(df['datetime'], errors='coerce').dropna()
        if dt.empty:
            continue

        # Make all timestamps comparable in one timezone
        if dt.dt.tz is None:
            dt = dt.dt.tz_localize(timezone)
        else:
            dt = dt.dt.tz_convert(timezone)

        start_times.append(dt.min())
        end_times.append(dt.max())

    if not start_times or not end_times:
        raise ValueError("No valid datetime ranges found in data_pkl.signal_data.")

    # Compute common start, end, and zoom window
    overlap_start_time = max(start_times)
    overlap_end_time = min(end_times)
    min_start_time = min(start_times)
    max_end_time = max(end_times)
    midpoint = overlap_start_time + (overlap_end_time - overlap_start_time) / 2
    zoom_window_start, zoom_window_end = midpoint - timedelta(minutes=zoom_time_window / 2), midpoint + timedelta(minutes=zoom_time_window / 2)

    # Update settings
    time_settings = {
        "overlap_start_time": str(overlap_start_time),
        "overlap_end_time": str(overlap_end_time),
        "zoom_window_start_time": str(zoom_window_start),
        "zoom_window_end_time": str(zoom_window_end),
    }
    param_manager.add_to_config(entries=time_settings, section="settings")

if any(v is None for v in time_settings.values()):
    print("YES")
time_settings

print(f"earliest logger start: {min_start_time}")
print(f"latest logger end: {max_end_time}")

In [ ]:
overlap_start_time

In [ ]:
overlap_end_time

In [ ]:
time_settings

In [ ]:
print(f"earliest logger start: {min_start_time}")
print(f"latest logger end: {max_end_time}")
overlap_end_time-overlap_start_time

In [ ]:
# # Plot arterial pO2 vs datetime
# import plotly.express as px

# df_o2 = data_pkl.signal_data['o2_pressure']

# # If datetime is the index, move it to a column for plotting
# if 'datetime' not in df_o2.columns and isinstance(df_o2.index, pd.DatetimeIndex):
#     df_o2 = df_o2.reset_index().rename(columns={df_o2.index.name or 'index': 'datetime'})

# # Ensure datetime column is datetime dtype
# df_o2['datetime'] = pd.to_datetime(df_o2['datetime'])

# fig = px.line(
#     df_o2,
#     x='datetime',
#     y='o2_pressure_arterial',
#     title=f"{deployment_id}: arterial pO2 over time",
#     labels={'datetime': 'Datetime', 'o2_pressure_arterial': 'pO2 arterial'}
# )
# fig.update_layout(hovermode='x unified')
# fig.show()

In [ ]:
data_pkl.event_data['datetime']

In [ ]:
data_pkl.signal_data['pressure']

In [ ]:
start = pd.Timestamp(time_settings['overlap_start_time'])# - timedelta(hours = 30)
end = pd.Timestamp(time_settings['overlap_end_time'])# + timedelta(hours=30)

fig = plot_tag_data_interactive(
    data_pkl=data_pkl,
    #signals = ['pressure','pitch','roll','heading'],
    # signals=['ecg', 'pressure'], # 'eeg', 'accelerometer', 'accelerometer2','prh'],
    #time_range=(start, end),
    note_annotations={"dive": {"signal": "depth", "symbol": "triangle-down", "color": "blue"}},
    state_annotations={"dive": {"signal": "depth", "color": "rgba(150, 150, 150, 0.3)"}},
    zoom_range_selector_channel='ecg',
    color_mapping_path=color_mapping_path,
    target_sampling_rate=25,
    state_annotation_channel_mode="combined",
    state_annotation_channel_height_ratio=0.2,
    state_annotation_channel_line_width=3.0,
)
fig.for_each_xaxis(
    lambda ax: ax.update(
        rangeslider=dict(visible=False),
        rangeselector=dict(visible=False),
    )
)
fig.show_dash(mode="inline")

In [ ]:
overwrite_data = True
pkl_path = os.path.join(deployment_folder, "outputs", "data.pkl")

effective_deployment_info = deployment_info
if manual_procedure_overrides_enabled:
    effective_deployment_info = apply_manual_procedure_overrides(
        deployment_info,
        manual_procedure_overrides,
    )
    print("🧪 Applied manual procedure overrides for:", list(manual_procedure_overrides.keys()))
else:
    print("ℹ️ Using procedure metadata extracted from Notion without notebook overrides.")

if os.path.exists(pkl_path) and not overwrite_data:
    with open(pkl_path, "rb") as f:
        data_pkl = pickle.load(f)
    print(f"📦 Loaded processed DataReader object from: {pkl_path}")
else:
    # Create outputs folder before initializing DataReader
    os.makedirs(os.path.join(deployment_folder, 'outputs'), exist_ok=True)
    
    # Initialize DataReader - it will automatically find deployment folder with suffix
    data_pkl = DataReader(
        dataset_folder=dataset_folder,
        deployment_id=deployment_id,
        data_subfolder="01_raw-data",
        montage_path=montage_path
    )
    
    # Initialize config manager
    param_manager = ParamManager(deployment_folder=deployment_folder, deployment_id=deployment_id)
    param_manager.add_to_config("current_processing_step", "Processing Step 00: Data import pending.")

    # Read and process all files
    data_pkl.read_files(
        deployment_info=effective_deployment_info,
        loggers_used=loggers_used,
        save_parq=False,
        save_netcdf=True
    )


In [ ]:
# optional save
pkl_path = os.path.join(deployment_folder, 'outputs', 'data.pkl')
with open(pkl_path, "wb") as file:
    pickle.dump(data_pkl, file)

In [ ]:
import xarray as xr

# Step 8: Update processing step
param_manager.add_to_config("current_processing_step", "Processing Step 00: Data imported.")

# Step 9: Open NetCDF file
netcdf_path = os.path.join(deployment_folder, "outputs", f'{deployment_id}_00_processed.nc')
if os.path.exists(netcdf_path):
    data = xr.open_dataset(netcdf_path)
    print(f"📊 NetCDF file loaded: {netcdf_path}")
else:
    print(f"⚠ NetCDF file not found at {netcdf_path}.")

data

In [ ]:
# Check if selected start and end times exist in the config file
truncate_times = param_manager.get_from_config(
    ["selected_start_time", "selected_end_time"],
    section="settings"
)

truncate_times

In [ ]:
if not any(v is None for v in truncate_times.values()):
    print("Truncating with provided cropping times.")
    # Update overlap window with selected range
    OVERLAP_START_TIME = pd.Timestamp(truncate_times['selected_start_time']).tz_convert(timezone)
    OVERLAP_END_TIME = pd.Timestamp(truncate_times['selected_end_time']).tz_convert(timezone)

    # Truncate signal data
    for signal, df in data_pkl.signal_data.items():
        # Truncate based on selected time range
        truncated_df = df[(df.iloc[:, 0] >= OVERLAP_START_TIME) & (df.iloc[:, 0] <= OVERLAP_END_TIME)].copy()
        data_pkl.signal_data[signal] = truncated_df  # Save truncated version to new variable

    # Recalculate Zoom Window (5-minute window in the middle)
    midpoint = OVERLAP_START_TIME + (OVERLAP_END_TIME - OVERLAP_START_TIME) / 2
    ZOOM_WINDOW_START_TIME = midpoint - timedelta(minutes=2.5)
    ZOOM_WINDOW_END_TIME = midpoint + timedelta(minutes=2.5)

    # Save new time settings
    time_settings_update = {
        "overlap_start_time": str(OVERLAP_START_TIME),
        "overlap_end_time": str(OVERLAP_END_TIME),
        "zoom_window_start_time": str(ZOOM_WINDOW_START_TIME),
        "zoom_window_end_time": str(ZOOM_WINDOW_END_TIME)
    }
    param_manager.add_to_config(entries=time_settings_update, section="settings")

    pkl_path = os.path.join(deployment_folder, 'outputs', 'data.pkl')
    with open(pkl_path, "wb") as file:
        pickle.dump(data_pkl, file)

In [ ]:
event_keys = sorted(
    data_pkl.event_data["key"].dropna().astype(str).unique().tolist()
) if hasattr(data_pkl, "event_data") and "key" in data_pkl.event_data else []

signals = list(data_pkl.signal_data.keys())
if "ecg" in signals:
    signals = ["ecg"] + [s for s in signals if s != "ecg"]

depth_signal = "depth" if "depth" in signals else (signals[0] if signals else None)
default_signal = depth_signal or (signals[0] if signals else None)

# Edit these mappings as needed. Keep only keys you want to preview.
state_annotations = {
    key: {"signal": depth_signal}
    for key in event_keys
    if depth_signal is not None and "dive" in key.lower()
}

note_annotations = {
    key: {"signal": default_signal}
    for key in event_keys
    if default_signal is not None and key not in state_annotations
}

preview_time_range = None
if "OVERLAP_START_TIME" in globals() and "OVERLAP_END_TIME" in globals():
    preview_time_range = (OVERLAP_START_TIME, OVERLAP_END_TIME)

fig = plot_tag_data_interactive(
    data_pkl=data_pkl,
    signals=signals,
    preserve_signal_order=True,
    time_range=preview_time_range,
    zoom_start_time=globals().get("ZOOM_WINDOW_START_TIME"),
    zoom_end_time=globals().get("ZOOM_WINDOW_END_TIME"),
    note_annotations=note_annotations,
    state_annotations=state_annotations,
    color_mapping_path=color_mapping_path,
    target_sampling_rate=1,
    state_annotation_channel_mode="combined",
    state_annotation_channel_height_ratio=0.2,
    state_annotation_channel_line_width=3.0,
)

fig.for_each_xaxis(
    lambda ax: ax.update(
        rangeslider=dict(visible=False),
        rangeselector=dict(visible=False),
    )
)

print("Preview state keys:", sorted(state_annotations.keys()))
print("Preview note keys:", sorted(note_annotations.keys()))
fig.show_dash(mode="inline")

## Inspect data

In [ ]:
# Load the data_reader object from the pickle file
pkl_path = os.path.join(deployment_folder, 'outputs', 'data.pkl')

with open(pkl_path, 'rb') as file:
    data_pkl = pickle.load(file)

for logger_id, info in data_pkl.logger_info.items():
    sampling_frequency = info.get('datetime_metadata', {}).get('fs', None)
    if sampling_frequency is not None:
        # Format the sampling frequency to 5 significant digits
        print(f"Sampling frequency for {logger_id}: {sampling_frequency} Hz")
    else:
        print(f"No sampling frequency available for {logger_id}")